## Importing Packages

In [18]:
import pandas as pd
import numpy as np
import polars as pl
import glob

import matplotlib.pyplot as plt
import seaborn as sns
import os

In [19]:
!rm -r tvDatafeed


In [20]:
import os
os.makedirs("tvDatafeed", exist_ok=True)


In [21]:
open("tvDatafeed/__init__.py", "w").close()


In [57]:
code = """
import pandas as pd
import warnings
from enum import Enum
from datetime import datetime, timedelta

try:
    import yfinance as yf
except:
    yf = None

class Interval(Enum):
    in_1_minute = "1m"
    in_3_minute = "3m"
    in_5_minute = "5m"
    in_15_minute = "15m"
    in_30_minute = "30m"
    in_60_minute = "60m"
    in_daily = "1d"
    in_1_day = "1d"
    in_weekly = "1wk"
    in_monthly = "1mo"

def _map_interval(interval):
    if isinstance(interval, Interval):
        return interval.value
    return str(interval)

def _to_yf_symbol(symbol, exchange):
    symbol = symbol.upper()
    exchange = exchange.upper()
    if exchange == "NSE":
        return symbol + ".NS"
    if exchange == "BSE":
        return symbol + ".BO"
    return symbol

class TvDatafeed:
    def __init__(self, username=None, password=None):
        if yf is None:
            raise ImportError("yfinance is required but not installed.")
        self.username = username
        self.password = password

    def get_hist(self, symbol, exchange="NSE", interval=Interval.in_daily, n_bars=1000, fut=False):
        yf_symbol = _to_yf_symbol(symbol, exchange)
        yf_interval = _map_interval(interval)

        # map n_bars to period (yfinance requires period for minute data)
        if yf_interval.endswith("m"):
            period = "60d"    # max allowed period for intraday
        else:
            years = max(1, int(n_bars / 250))
            period = f"{years}y"

        try:
            df = yf.download(
                tickers=yf_symbol,
                period=period,
                interval=yf_interval,
                progress=False,
                auto_adjust=True # Explicitly set auto_adjust to True to prevent MultiIndex columns
            )
        except Exception as e:
            raise RuntimeError(f"Data fetch failed: {e}")

        if df is None or df.empty:
            raise RuntimeError(f"No data returned for {symbol} from yfinance.")

        df.index.name = "date"
        # Convert column names to lowercase and flatten if they are MultiIndex tuples
        df.columns = [c[0].lower() if isinstance(c, tuple) else c.lower() for c in df.columns]

        # Ensure all required OHLCV columns exist and handle missing ones
        keep = ["open", "high", "low", "close", "volume"]
        for k in keep:
            if k not in df.columns:
                df[k] = None # Fill with None if a column is genuinely missing (unlikely for OHLCV from yfinance)

        # Tail n_bars
        df = df[keep].tail(n_bars)
        return df

"""

with open("tvDatafeed/__init__.py", "w") as f:
    f.write(code)

print("tvDatafeed installed!")

tvDatafeed installed!


In [58]:
import sys
sys.modules.pop('tvDatafeed', None)

<module 'tvDatafeed' from '/content/tvDatafeed/__init__.py'>

In [59]:
from tvDatafeed import TvDatafeed, Interval
print("SUCCESS!", TvDatafeed, Interval.in_daily)

SUCCESS! <class 'tvDatafeed.TvDatafeed'> Interval.in_daily


## Creating the Dataset

### Logging into TVFeed

In [60]:
tv = TvDatafeed(username = 'jaganathapandiyan12', password = 'PASS$1234TO5678')

### Downloading the stocks in NIFTY 500 (list downloaded from NSE website on 16th November)

In [26]:
data_nifty500 = pd.read_csv('ind_nifty500list.csv')

symbols_nifty500 = data_nifty500['Symbol'].to_list()
symbols_nifty500 = [s.replace("-", "_") for s in symbols_nifty500]
symbols_nifty500.remove('DUMMYSKFIN')


print(f'Total stocks: {len(symbols_nifty500)}')

Total stocks: 501


In [27]:
import pandas as pd

df = pd.read_csv("ind_nifty500list.csv")

symbols_raw = df['Symbol'].astype(str).str.strip().tolist()

symbols_clean = []
for s in symbols_raw:
    s = s.upper().strip()

    # Skip empty values or invalid
    if not s or s == 'nan':
        continue

    # Prevent double .NS
    if s.endswith(".NS"):
        symbols_clean.append(s)
    else:
        symbols_clean.append(s + ".NS")

print("Sample cleaned symbols:", symbols_clean[:10])


Sample cleaned symbols: ['360ONE.NS', '3MINDIA.NS', 'ABB.NS', 'ACC.NS', 'ACMESOLAR.NS', 'AIAENG.NS', 'APLAPOLLO.NS', 'AUBANK.NS', 'AWL.NS', 'AADHARHFC.NS']


In [61]:
failed = []

import os, time
os.makedirs("data/raw/prices", exist_ok=True)

for symbol in symbols_clean:
    try:
        df = tv.get_hist(
            symbol=symbol.replace(".NS","",1), # Use replace count=1 to avoid issues if .NS appears elsewhere
            exchange='NSE',
            interval=Interval.in_daily,
            n_bars=2000
        )

        if df is None or df.empty:
            raise ValueError(f"No data returned for {symbol}")

        outname = symbol.replace(".NS", "",1)
        df.to_parquet(f"data/raw/prices/{outname}.parquet")

        print("✔ Downloaded:", symbol)

    except Exception as e:
        print("✖ Failed:", symbol, "→", e)
        failed.append(symbol)

    time.sleep(0.05)

✔ Downloaded: 360ONE.NS
✔ Downloaded: 3MINDIA.NS
✔ Downloaded: ABB.NS
✔ Downloaded: ACC.NS
✔ Downloaded: ACMESOLAR.NS
✔ Downloaded: AIAENG.NS
✔ Downloaded: APLAPOLLO.NS
✔ Downloaded: AUBANK.NS
✔ Downloaded: AWL.NS
✔ Downloaded: AADHARHFC.NS
✔ Downloaded: AARTIIND.NS
✔ Downloaded: AAVAS.NS
✔ Downloaded: ABBOTINDIA.NS
✔ Downloaded: ACE.NS
✔ Downloaded: ADANIENSOL.NS
✔ Downloaded: ADANIENT.NS
✔ Downloaded: ADANIGREEN.NS
✔ Downloaded: ADANIPORTS.NS
✔ Downloaded: ADANIPOWER.NS
✔ Downloaded: ATGL.NS
✔ Downloaded: ABCAPITAL.NS
✔ Downloaded: ABFRL.NS
✔ Downloaded: ABLBL.NS
✔ Downloaded: ABREL.NS
✔ Downloaded: ABSLAMC.NS
✔ Downloaded: ADVENTHTL.NS
✔ Downloaded: AEGISLOG.NS
✔ Downloaded: AEGISVOPAK.NS
✔ Downloaded: AFCONS.NS
✔ Downloaded: AFFLE.NS
✔ Downloaded: AJANTPHARM.NS
✔ Downloaded: AKUMS.NS
✔ Downloaded: AKZOINDIA.NS
✔ Downloaded: APLLTD.NS
✔ Downloaded: ALKEM.NS
✔ Downloaded: ALKYLAMINE.NS
✔ Downloaded: ALOKINDS.NS
✔ Downloaded: ARE&M.NS
✔ Downloaded: AMBER.NS
✔ Downloaded: AMBUJACEM.NS


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['DUMMYSKFIN.NS']: YFPricesMissingError('possibly delisted; no price data found  (period=8y) (Yahoo error = "No data found, symbol may be delisted")')


✔ Downloaded: DRREDDY.NS
✖ Failed: DUMMYSKFIN.NS → No data returned for DUMMYSKFIN from yfinance.
✔ Downloaded: EIDPARRY.NS
✔ Downloaded: EIHOTEL.NS
✔ Downloaded: EICHERMOT.NS
✔ Downloaded: ELECON.NS
✔ Downloaded: ELGIEQUIP.NS
✔ Downloaded: EMAMILTD.NS
✔ Downloaded: EMCURE.NS
✔ Downloaded: ENDURANCE.NS
✔ Downloaded: ENGINERSIN.NS
✔ Downloaded: ERIS.NS
✔ Downloaded: ESCORTS.NS
✔ Downloaded: ETERNAL.NS
✔ Downloaded: EXIDEIND.NS
✔ Downloaded: NYKAA.NS
✔ Downloaded: FEDERALBNK.NS
✔ Downloaded: FACT.NS
✔ Downloaded: FINCABLES.NS
✔ Downloaded: FINPIPE.NS
✔ Downloaded: FSL.NS
✔ Downloaded: FIVESTAR.NS
✔ Downloaded: FORCEMOT.NS
✔ Downloaded: FORTIS.NS
✔ Downloaded: GAIL.NS
✔ Downloaded: GVT&D.NS
✔ Downloaded: GMRAIRPORT.NS
✔ Downloaded: GRSE.NS
✔ Downloaded: GICRE.NS
✔ Downloaded: GILLETTE.NS
✔ Downloaded: GLAND.NS
✔ Downloaded: GLAXO.NS
✔ Downloaded: GLENMARK.NS
✔ Downloaded: MEDANTA.NS
✔ Downloaded: GODIGIT.NS
✔ Downloaded: GPIL.NS
✔ Downloaded: GODFRYPHLP.NS
✔ Downloaded: GODREJAGRO.NS
✔ Do

In [38]:
import glob
import pandas as pd

all_files = glob.glob("data/raw/prices/*.parquet")
dfs = []

for f in all_files:
    symbol = f.split("/")[-1].replace(".parquet","")
    temp = pd.read_parquet(f)
    temp["symbol"] = symbol
    dfs.append(temp)

df_all = pd.concat(dfs)
df_all.to_parquet("data/processed/Prices_all.parquet")

print("Processed file created:", "data/processed/Prices_all.parquet")


Processed file created: data/processed/Prices_all.parquet


In [12]:
!git clone --branch Rokith_algo https://github.com/jagan010101/Algo-Trading.git


Cloning into 'Algo-Trading'...
fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Merging the files into a single file for analysis

In [62]:
import pandas as pd
import polars as pl
import os
import glob
import pyarrow # pandas will use it if available for to_parquet engine

RAW_DIR = "data/raw/prices"
OUT_FILE = "data/Processed/Prices_all.parquet"

def merge_parquet_files():
    # Remove the existing file if it's corrupted or incomplete
    if os.path.exists(OUT_FILE):
        os.remove(OUT_FILE)
        print(f"Removed existing corrupted file: {OUT_FILE}")

    files = glob.glob(os.path.join(RAW_DIR, "*.parquet"))

    if len(files) == 0:
        print("No parquet files found in data/raw/prices/")
        return

    dfs = []

    for f in files:
        symbol = os.path.basename(f).replace(".parquet", "")
        try:
            df_polars = (pl.read_parquet(f).with_columns([pl.lit(symbol).alias("symbol"),
                                                          pl.col("date").cast(pl.Date).alias("date")
                                                         ]))
            dfs.append(df_polars)
        except Exception as e:
            print(f"Error reading individual parquet file {f}: {e}")
            continue

    if not dfs:
        print("No valid dataframes to merge.")
        return

    final_df_polars = pl.concat(dfs, how="vertical")
    final_df_polars = final_df_polars.sort(["symbol", "date"])

    os.makedirs(os.path.dirname(OUT_FILE), exist_ok=True)

    try:
        # Convert to pandas DataFrame and write using pandas' to_parquet
        # This will use pyarrow or fastparquet engine if available.
        # pyarrow is usually installed with polars environments.
        final_df_pandas = final_df_polars.to_pandas()
        final_df_pandas.to_parquet(OUT_FILE, engine='pyarrow', compression='snappy')

        print(f"Master datafile created using pandas.to_parquet: {OUT_FILE}")
        print(f"Rows: {final_df_polars.height}, Columns: {final_df_polars.width}")

        # Verification step: Try to read the file immediately after writing with Polars
        try:
            _ = pl.read_parquet(OUT_FILE)
            print(f"Verification successful: {OUT_FILE} can be read back by Polars.")
        except Exception as e:
            print(f"Verification failed for {OUT_FILE} with Polars read, even after pandas write: {e}")
            if os.path.exists(OUT_FILE):
                os.remove(OUT_FILE)
            print(f"Removed problematic file: {OUT_FILE}")
            return

    except Exception as e:
        print(f"Error writing parquet file {OUT_FILE} using pandas.to_parquet: {e}")
        if os.path.exists(OUT_FILE):
            os.remove(OUT_FILE)
        return


if __name__ == "__main__":
    merge_parquet_files()

Removed existing corrupted file: data/Processed/Prices_all.parquet
Master datafile created using pandas.to_parquet: data/Processed/Prices_all.parquet
Rows: 832458, Columns: 7
Verification successful: data/Processed/Prices_all.parquet can be read back by Polars.


## Calculating the Required Ratios

In [63]:
df = pl.read_parquet("data/Processed/Prices_all.parquet", use_pyarrow=True)
df = df.sort(["symbol", "date"])

### 1) MACD (12-day EMA - 26-day EMA)

In [90]:
df = df.with_columns([
    pl.col("close").ewm_mean(span = 12).over("symbol").alias("ema_12"),
    pl.col("close").ewm_mean(span = 26).over("symbol").alias("ema_26"),]).with_columns([
    (pl.col("ema_12") - pl.col("ema_26")).alias("macd")])

### 2) 14-day ROC

In [91]:
df = df.with_columns([
    ((pl.col("close") - pl.col("close").shift(14))
     / pl.col("close").shift(14)).over("symbol").alias("roc_14")])

### 3) 14-day ADX

In [92]:
df = df.with_columns([
    pl.col("high").shift(1).over("symbol").alias("prev_high"),
    pl.col("low").shift(1).over("symbol").alias("prev_low"),
    pl.col("close").shift(1).over("symbol").alias("prev_close"),])

df = df.with_columns([
    (pl.col("high") - pl.col("low")).alias("hl_range"),
    (pl.col("high") - pl.col("prev_close")).abs().alias("hc_range"),
    (pl.col("low") - pl.col("prev_close")).abs().alias("lc_range"),])

df = df.with_columns([
    pl.max_horizontal([
        pl.col("hl_range"),
        pl.col("hc_range"),
        pl.col("lc_range"),
    ]).alias("tr")])

df = df.with_columns([
    (pl.col("high") - pl.col("prev_high")).alias("up_move"),
    (pl.col("prev_low") - pl.col("low")).alias("down_move"),])

df = df.with_columns([
    pl.when(
        (pl.col("up_move") > pl.col("down_move")) & (pl.col("up_move") > 0)
        ).then(pl.col("up_move")).otherwise(0).alias("plus_dm"),

    pl.when(
        (pl.col("down_move") > pl.col("up_move")) & (pl.col("down_move") > 0)
        ).then(pl.col("down_move")).otherwise(0).alias("minus_dm"),])

df = df.with_columns([
    pl.col("tr").rolling_mean(14).over("symbol").alias("atr_14"),
    pl.col("plus_dm").rolling_mean(14).over("symbol").alias("plus_dm_14"),
    pl.col("minus_dm").rolling_mean(14).over("symbol").alias("minus_dm_14"),])


df = df.with_columns([
    (100 * pl.col("plus_dm_14") / pl.col("atr_14")).alias("plus_di_14"),
    (100 * pl.col("minus_dm_14") / pl.col("atr_14")).alias("minus_di_14"),])


df = df.with_columns([
    (100 * (pl.col("plus_di_14") - pl.col("minus_di_14")).abs()
     / (pl.col("plus_di_14") + pl.col("minus_di_14"))
    ).alias("dx_14")])

df = df.with_columns([
    pl.col("dx_14").rolling_mean(14).over("symbol").alias("adx_14")])

### 4) 5-day VWAP

In [93]:
df = df.with_columns(
    ((pl.col("high") + pl.col("low") + pl.col("close")) / 3).alias("typical_price"))

df = df.with_columns([
    (pl.col("typical_price") * pl.col("volume")).rolling_sum(5).over("symbol").alias("tp_vol_sum_5"),
    pl.col("volume").rolling_sum(5).over("symbol").alias("vol_sum_5"),
                    ]).with_columns([(pl.col("tp_vol_sum_5") / pl.col("vol_sum_5")).alias("vwap_5")])

### 5) 14-day RSI

In [94]:
df = df.with_columns(
    pl.col("close").diff().over("symbol").alias("delta"))

df = df.with_columns([
    pl.col("delta").clip(lower_bound=0).alias("gain"),
    (-pl.col("delta").clip(upper_bound=0)).alias("loss")])

df = df.with_columns([
    pl.col("gain").rolling_mean(14).over("symbol").alias("avg_gain_14"),
    pl.col("loss").rolling_mean(14).over("symbol").alias("avg_loss_14")])

df = df.with_columns((
    100 - 100 / (1 + (pl.col("avg_gain_14") / pl.col("avg_loss_14")))).alias("rsi_14"))

### 6) 20-day Volume

In [95]:
df = df.with_columns(
    pl.col("volume").rolling_mean(20).over("symbol").alias("sma_vol_20"))

### 7) 14-day ATR

In [96]:
df = df.with_columns([
    pl.col("close").shift(1).over("symbol").alias("prev_close")])

df = df.with_columns([
    pl.max_horizontal([pl.col("high") - pl.col("low"),
                       (pl.col("high") - pl.col("prev_close")).abs(),
                       (pl.col("low")  - pl.col("prev_close")).abs()]).alias("true_range")])

df = df.with_columns([
    pl.col("true_range").rolling_mean(14).over("symbol").alias("atr_14")])

8) Daily Return

In [97]:
df = df.with_columns(
    (
        pl.col("close") / pl.col("close").shift(1) - 1
    )
    .over("symbol")
    .alias("daily_ret")
)

9) 60 Day Histotical volatility

In [98]:
df = df.with_columns(
    (
        pl.col("daily_ret")
        .rolling_std(60)
        .over("symbol")
        * (252 ** 0.5)
    )
    .alias("hv60")
)

### Dropping unnecessary columns and removing NULL rows

In [99]:
df = df.drop([
    "ema_12", "ema_26", "delta", "gain", "loss", "avg_gain_14", "avg_loss_14", "typical_price",
    "tp_vol_sum_5", "vol_sum_5", "prev_close", "true_range", "prev_high", "prev_low",
    "hl_range", "hc_range", "lc_range", "up_move", "down_move", "plus_dm", "minus_dm", "plus_dm_14",
    "minus_dm_14", "dx_14", "tr", "plus_di_14", "minus_di_14"
], strict=False)

In [100]:
print(f'Before dropping nulls: {df.shape}')

df = df.drop_nulls()

print(f'After dropping nulls: {df.shape}')

Before dropping nulls: (784526, 17)
After dropping nulls: (754807, 17)


In [101]:
df.head()

open,high,low,close,volume,date,symbol,atr_14,adx_14,vwap_5,rsi_14,sma_vol_20,macd,daily_ret,hv60,forward_return_5d,roc_14
f64,f64,f64,f64,i64,datetime[ms],str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
192.798306,192.798306,180.972098,182.704529,11816,2020-04-27 00:00:00,"""360ONE""",14.405595,21.1087,185.470118,33.888013,151184.4,-14.181195,-0.010784,0.742243,0.033216,-0.135862
180.972151,194.821267,176.148302,186.45993,39668,2020-04-28 00:00:00,"""360ONE""",14.171441,19.283043,186.153432,40.889226,146422.6,-13.627282,0.020555,0.74412,-0.017637,-0.075078
180.961773,192.850224,180.961773,184.084305,133420,2020-04-29 00:00:00,"""360ONE""",14.304821,17.457386,186.65109,36.725344,149914.4,-13.225998,-0.012741,0.744261,-0.009862,-0.105008
186.522149,201.253041,186.522149,197.736298,74956,2020-04-30 00:00:00,"""360ONE""",13.806134,17.58739,188.502755,41.541017,54989.0,-11.679044,0.074162,0.76171,-0.081161,-0.070059
192.030673,215.309636,186.968233,190.318985,139904,2020-05-04 00:00:00,"""360ONE""",14.525636,17.324919,191.709955,39.990935,60932.4,-10.921917,-0.037511,0.763501,-0.061103,-0.087763


## Setting up the Strategies

## Stock selection

FILTERING HIGH VOLATILE AND LOW VOLUME STOCKS

In [86]:
# Most recent available values per stock
latest = df.sort('date').group_by('symbol').tail(1)
MIN_VOLUME = 300000      # stocks with avg volume < 3 lakh are removed
MAX_VOLATILITY = 0.60    # stocks with > 60% annual vol removed
filtered = latest.filter(
    (latest['sma_vol_20'] > MIN_VOLUME) &
    (latest['hv60'] < MAX_VOLATILITY)
)
# Get new filtered Dataframe
filtered_symbols = filtered['symbol'].unique().to_list()
print("Total stocks after filtering:", len(filtered_symbols))

Total stocks after filtering: 360


XG-BOOST REGRESSOR CONSTRUCTION

In [103]:
import xgboost as xgb
import numpy as np # Import numpy for np.inf

# Calculate a forward 5-day return as the target variable
df = df.with_columns(
    (pl.col("close").shift(-5).over("symbol") / pl.col("close") - 1).alias("forward_return_5d")
)

# Drop rows where the target or features might be null due to shifting or rolling calculations
df = df.drop_nulls()

# Replace inf values with NaN to prevent XGBoost errors
for col in df.columns:
    if df[col].dtype.is_float():
        df = df.with_columns(pl.col(col).replace(float('inf'), np.nan).replace(float('-inf'), np.nan))

# Drop nulls again after replacing inf with NaN
df = df.drop_nulls()

# Print columns for debugging
print("Columns in Polars DataFrame before pandas conversion:", df.columns)

# Split into train & test (time-based)
# Convert to pandas for XGBoost compatibility and simpler time-based split
df_pandas = df.to_pandas()

train = df_pandas[df_pandas['date'] < "2023-01-01"]
test  = df_pandas[df_pandas['date'] >= "2023-01-01"]

# ---------------------------------------------------------
# DEFINE FEATURE SETS (using available calculated features)
# ---------------------------------------------------------

features = [
    'macd', 'roc_14', 'adx_14', 'vwap_5', 'rsi_14', 'sma_vol_20', 'hv60', 'atr_14', 'daily_ret'
]

target = "forward_return_5d"


# ---------------------------------------------------------
# TRAIN XGBOOST REGRESSOR
# ---------------------------------------------------------

def train_xgboost(X, y):
    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "max_depth": 6,
        "eta": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "n_estimators": 300,
        "n_jobs": -1 # Use all available cores
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X, y)
    return model

# Train a single model
model = train_xgboost(train[features], train[target])


# ---------------------------------------------------------
#  PREDICT EXPECTED RETURN
# ---------------------------------------------------------

test["predicted_return"]  = model.predict(test[features])


# ---------------------------------------------------------
# SELECT TOP-N STOCKS
# ---------------------------------------------------------

def select_top_n(df_input, prediction_column, n=10, min_thresh=0.0):
    """Returns top N stocks by expected return above threshold."""
    # Ensure 'symbol' and prediction_column exist
    if 'symbol' not in df_input.columns or prediction_column not in df_input.columns:
        raise ValueError(f"DataFrame must contain 'symbol' and '{prediction_column}' columns")

    # Convert to Polars for group_by and sort for efficiency if not already
    if isinstance(df_input, pd.DataFrame):
        df_polars = pl.from_pandas(df_input)
    else:
        df_polars = df_input

    # Group by symbol and calculate mean of prediction_column
    temp = df_polars.group_by('symbol').agg(
        pl.col(prediction_column).mean().alias('avg_predicted_return')
    )

    # Filter by min_thresh
    temp = temp.filter(pl.col('avg_predicted_return') > min_thresh)

    # Sort and select top N
    return temp.sort('avg_predicted_return', descending=True).head(n).to_pandas()


TopN_stocks  = select_top_n(test, "predicted_return", n=10, min_thresh=0.01)

print("\n===== TOP STOCKS BASED ON PREDICTED RETURN ====")
print(TopN_stocks)


# ---------------------------------------------------------
# EXPORT OUTPUT FOR EXECUTION ENGINE
# ---------------------------------------------------------

final_selection = TopN_stocks.assign(strategy="XGBOOST")
final_selection.to_csv("selected_stocks_for_execution.csv", index=False)

print("\nSaved: selected_stocks_for_execution.csv")

Columns in Polars DataFrame before pandas conversion: ['open', 'high', 'low', 'close', 'volume', 'date', 'symbol', 'atr_14', 'adx_14', 'vwap_5', 'rsi_14', 'sma_vol_20', 'macd', 'daily_ret', 'hv60', 'forward_return_5d', 'roc_14']

===== TOP STOCKS BASED ON PREDICTED RETURN ====
    symbol  avg_predicted_return
0  YESBANK              0.019130
1     CGCL              0.011492
2     GPIL              0.010878
3     ATGL              0.010012

Saved: selected_stocks_for_execution.csv


/tmp/ipython-input-1199875428.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test["predicted_return"]  = model.predict(test[features])


## Performance Assessment

## Portfolio Reallocation

## Backtesting